# U.S. State-Level Real GDP Data Preparation and Panel Construction

This notebook retrieves and prepares the quarterly U.S. state real GDP dataset from the U.S. Bureau of Economic Analysis (BEA) Regional Economic Accounts API. The data are cleaned and structured into a State–Year–Quarter panel format. The resulting dataset is used as an explanatory variable for structural break and recovery analysis in the Capstone Project.

In [150]:
import pandas as pd
import numpy as np
import requests
import prettytable
import sqlite3
from pathlib import Path

prettytable.DEFAULT = 'DEFAULT'

In [151]:
# --- PATH CONFIG  ---
from pathlib import Path

current = Path().resolve()
while current != current.parent:
    if (current / "Data").exists():
        PROJECT_ROOT = current
        break
    current = current.parent

DATA_RAW = PROJECT_ROOT / "Data" / "Raw"
DATA_CLEAN = PROJECT_ROOT / "Data" / "Cleaned"
DATA_PROCESSED = PROJECT_ROOT / "Data" / "Processed"
DB_PATH = PROJECT_ROOT / "Capstone.db"

print("Project root:", PROJECT_ROOT)

Project root: /Users/emerino/Documents/Capstone_Project_Group_4-


## 1) Retrieve state real GDP data from BEA API

In [152]:
# API Key BEA
BEA_KEY = "2394E938-2D17-4899-97C3-4E9D2EE25FB6"

url = "https://apps.bea.gov/api/data"

params = {
    "UserID": BEA_KEY,
    "Method": "GetData",
    "datasetname": "Regional",
    "TableName": "SQGDP9",
    "LineCode": "1",  # Real GDP (chained dollars)
    "Year": "2015,2016,2017,2018,2019,2020,2021,2022,2023,2024",
    "GeoFips": "STATE",
    "ResultFormat": "json"
}

# Request
r = requests.get(url, params=params).json()

# To DataFrame
df = pd.DataFrame(r["BEAAPI"]["Results"]["Data"])

## 2) Filter to the 50 U.S. states

In [153]:
# Filter 50 states

states_50 = [
'Alabama','Alaska','Arizona','Arkansas','California','Colorado',
'Connecticut','Delaware','Florida','Georgia','Hawaii','Idaho',
'Illinois','Indiana','Iowa','Kansas','Kentucky','Louisiana',
'Maine','Maryland','Massachusetts','Michigan','Minnesota',
'Mississippi','Missouri','Montana','Nebraska','Nevada',
'New Hampshire','New Jersey','New Mexico','New York',
'North Carolina','North Dakota','Ohio','Oklahoma','Oregon',
'Pennsylvania','Rhode Island','South Carolina','South Dakota',
'Tennessee','Texas','Utah','Vermont','Virginia','Washington',
'West Virginia','Wisconsin','Wyoming'
]

df = df[df["GeoName"].isin(states_50)]

## 3) Convert GDP values to numeric

In [154]:
# convert to GDP
df["DataValue"] = df["DataValue"].str.replace(",", "", regex=False)
df["DataValue"] = pd.to_numeric(df["DataValue"], errors="coerce")

## 4) Extract year and quarter

In [155]:
# Separate year and quarter
df["year"] = df["TimePeriod"].str[:4].astype(int)
df["quarter"] = df["TimePeriod"].str[-1].astype(int)

## 5) Standardize column names and ordering

In [156]:
df = df.rename(columns={
    "GeoName": "state",
    "DataValue": "Real_GDP"
})

df = df.sort_values(["state", "year", "quarter"]).reset_index(drop=True)

## 6) Inspect dataset structure

In [157]:
print("Final states:", df["state"].nunique())
print(df.head())

Final states: 50
       Code GeoFips    state TimePeriod                           CL_UNIT  \
0  SQGDP9-1   01000  Alabama     2015Q1  Millions of chained 2017 dollars   
1  SQGDP9-1   01000  Alabama     2015Q2  Millions of chained 2017 dollars   
2  SQGDP9-1   01000  Alabama     2015Q3  Millions of chained 2017 dollars   
3  SQGDP9-1   01000  Alabama     2015Q4  Millions of chained 2017 dollars   
4  SQGDP9-1   01000  Alabama     2016Q1  Millions of chained 2017 dollars   

  UNIT_MULT  Real_GDP  year  quarter  
0         6  206947.5  2015        1  
1         6  208955.9  2015        2  
2         6  210102.0  2015        3  
3         6  209795.8  2015        4  
4         6  211142.4  2016        1  


## 7) Drop unnecessary columns

In [158]:
# Drop unnecessary columns

df = df.drop(columns=[
    "Code",
    "GeoFips",
    "CL_UNIT",
    "UNIT_MULT",
    "TimePeriod"
])

## 8) Convert GDP to billions

In [159]:
# Ensure GDP is numeric
df["GDP_billions"] = df["Real_GDP"] / 1000
df["GDP_billions"] = df["GDP_billions"].round(2)

In [160]:
df.head(5)

,state,Real_GDP,year,quarter,GDP_billions
0,Alabama,206947.5,2015,1,206.95
1,Alabama,208955.9,2015,2,208.96
2,Alabama,210102.0,2015,3,210.10
3,Alabama,209795.8,2015,4,209.80
4,Alabama,211142.4,2016,1,211.14


## 9) Handle missing values 

In [161]:
df.isna().sum()

state           0
Real_GDP        0
year            0
quarter         0
GDP_billions    0
dtype: int64

## 10) Build final GDP panel

In [162]:
df_clean = df[["state", "year", "quarter", "GDP_billions"]].copy()

# Convert quarterly GDP to monthly frequency
# Each quarter value will be repeated for its 3 corresponding months

# Map each quarter to its calendar months
quarter_to_months = {
    1: [1, 2, 3],
    2: [4, 5, 6],
    3: [7, 8, 9],
    4: [10, 11, 12]
}

# Repeat each row 3 times (one for each month in the quarter)
df_monthly = df_clean.loc[df_clean.index.repeat(3)].copy()

# Create a month position within each quarter (1,2,3)
df_monthly["month_in_quarter"] = (
    df_monthly.groupby(["state", "year", "quarter"])
    .cumcount() + 1
)

# Convert quarter-relative month to actual calendar month
df_monthly["month"] = df_monthly.apply(
    lambda r: quarter_to_months[r["quarter"]][r["month_in_quarter"] - 1],
    axis=1
)

# Drop helper column and quarter (no longer needed)
df_monthly = df_monthly.drop(columns=["month_in_quarter", "quarter"])

# Sort final dataset
df_monthly = df_monthly.sort_values(["state", "year", "month"]).reset_index(drop=True)

df_monthly = df_monthly[["state", "year", "month", "GDP_billions"]]

## 11) Export clean GDP panel

Merge keys you will use later:

- `State`, `Year`, `Month`

This will allow clean merges with:

- Motor bus ridership (state–month via quarter aggregation)
- BLS unemployment (state–month → quarter)
- policy stringency/mobility (state–month → quarter)
- fuel prices (region/state)
- red/blue classification (state-level)
- workforce composition (state-year)


In [163]:
# Remove Idaho
df_monthly = df_monthly[df_monthly["state"] != "Idaho"]
df_monthly = df_monthly[df_monthly["state"] != "Wyoming"]

# Path to Cleaned folder
DATA_CLEAN = PROJECT_ROOT / "Data" / "Cleaned"

# Export monthly GDP panel to Cleaned
output_path = DATA_CLEAN / "clean_state_gdp_2015_2024.csv"
df_monthly.to_csv(output_path, index=False)

print(f"Saved: {output_path}")
print("States:", df_monthly["state"].nunique())
print("Rows:", len(df_monthly))


Saved: /Users/emerino/Documents/Capstone_Project_Group_4-/Data/Cleaned/clean_state_gdp_2015_2024.csv
States: 48
Rows: 5760


In [164]:
states_48 = [
'Alabama','Alaska','Arizona','Arkansas','California','Colorado',
'Connecticut','Delaware','Florida','Georgia','Hawaii','Illinois',
'Indiana','Iowa','Kansas','Kentucky','Louisiana','Maine',
'Maryland','Massachusetts','Michigan','Minnesota',
'Mississippi','Missouri','Montana','Nebraska','Nevada',
'New Hampshire','New Jersey','New Mexico','New York',
'North Carolina','North Dakota','Ohio','Oklahoma','Oregon',
'Pennsylvania','Rhode Island','South Carolina','South Dakota',
'Tennessee','Texas','Utah','Vermont','Virginia','Washington',
'West Virginia','Wisconsin'
]

## 12) Load and Extract Only Needed Columns (Population)

In [165]:
# Load datasets
pop_old = pd.read_csv(DATA_RAW / "nst-est2020.csv")
pop_new = pd.read_csv(DATA_RAW / "NST-EST2025-ALLDATA.csv")

# Keep only state-level rows
pop_old = pop_old[pop_old["SUMLEV"] == 40]
pop_new = pop_new[pop_new["SUMLEV"] == 40]

# Keep only relevant columns
cols_old = [
    "NAME",
    "POPESTIMATE2015",
    "POPESTIMATE2016",
    "POPESTIMATE2017",
    "POPESTIMATE2018",
    "POPESTIMATE2019"
]

cols_new = [
    "NAME",
    "POPESTIMATE2020",
    "POPESTIMATE2021",
    "POPESTIMATE2022",
    "POPESTIMATE2023",
    "POPESTIMATE2024",
    "POPESTIMATE2025"
]

pop_old = pop_old[cols_old]
pop_new = pop_new[cols_new]

## 13) Merge Years Horizontally

In [166]:
# Merge by state name
pop = pd.merge(pop_old, pop_new, on="NAME", how="inner")

pop = pop.rename(columns={"NAME": "state"})

## 14) Keep Only 48 States

In [167]:
pop = pop[pop["state"].isin(states_48)].copy()

## 15) Reshape to Long Format

In [168]:
pop_long = pop.melt(
    id_vars="state",
    var_name="year",
    value_name="population"
)

pop_long["year"] = pop_long["year"].str[-4:].astype(int)

pop_long = pop_long.sort_values(["state", "year"]).reset_index(drop=True)

## 16) Interpolate Monthly Population

In [169]:
# Create next year population
pop_long["pop_next"] = pop_long.groupby("state")["population"].shift(-1)

# Drop 2025 
pop_long = pop_long[pop_long["year"] < 2026]

# Expand to monthly
pop_monthly = pop_long.loc[pop_long.index.repeat(12)].copy()

pop_monthly["month"] = (
    pop_monthly.groupby(["state", "year"])
    .cumcount() + 1
)

# Linear interpolation
pop_monthly["population_monthly"] = (
    pop_monthly["population"] +
    (pop_monthly["month"] - 1) / 12 *
    (pop_monthly["pop_next"] - pop_monthly["population"])
)

pop_monthly = pop_monthly[["state", "year", "month", "population_monthly"]]

## 17) Merge With Monthly GDP

In [170]:
gdp = pd.read_csv(DATA_CLEAN / "clean_state_gdp_2015_2024.csv")
gdp = gdp[gdp["state"].isin(states_48)]

df_final = pd.merge(
    gdp,
    pop_monthly,
    on=["state", "year", "month"],
    how="inner"
)

## 18) Calculate GDP Per Capita

In [171]:
df_final["gdp_per_capita"] = (
    df_final["GDP_billions"] * 1_000_000_000 /
    df_final["population_monthly"]
)

df_final["gdp_per_capita"] = df_final["gdp_per_capita"].round(2)

## 19) Final Structure

In [172]:
df_final = df_final[[
    "state",
    "year",
    "month",
    "GDP_billions",
    "population_monthly",
    "gdp_per_capita"
]]

## 20) Export Clean File

In [173]:
output_path = DATA_CLEAN / "clean_state_gdp_per_capita_2015_2024.csv"
df_final.to_csv(output_path, index=False)

print("Saved:", output_path)
print("States:", df_final["state"].nunique())
print("Rows:", len(df_final))

Saved: /Users/emerino/Documents/Capstone_Project_Group_4-/Data/Cleaned/clean_state_gdp_per_capita_2015_2024.csv
States: 48
Rows: 5760


## 21) Load GDP panel into SQLite database

In [174]:
# Connect to SQLite in project root
con = sqlite3.connect(PROJECT_ROOT / "Capstone.db")
cur = con.cursor()
%load_ext sql
%sql sqlite:///{PROJECT_ROOT}/Capstone.db

df_monthly = pd.read_csv(DATA_CLEAN / "clean_state_gdp_per_capita_2015_2024.csv")
df_monthly.to_sql("GDP_Percapita", con, if_exists='replace', index=False, method="multi")

print("GDP_Percapita table loaded from Cleaned folder")

The sql extension is already loaded. To reload it, use:
  %reload_ext sql
GDP_Percapita table loaded from Cleaned folder


## 22) Validate GDP database ingestion

In [175]:
%sql SELECT *FROM GDP_Percapita WHERE state= 'Florida' LIMIT 12;

 * sqlite:////Users/emerino/Documents/Capstone_Project_Group_4-/Capstone.db
Done.


state,year,month,GDP_billions,population_monthly,gdp_per_capita
Florida,2015,1,929.32,20219111.0,45962.46
Florida,2015,2,929.32,20253121.5,45885.27
Florida,2015,3,929.32,20287132.0,45808.35
Florida,2015,4,942.83,20321142.5,46396.51
Florida,2015,5,942.83,20355153.0,46318.98
Florida,2015,6,942.83,20389163.5,46241.72
Florida,2015,7,951.36,20423174.0,46582.38
Florida,2015,8,951.36,20457184.5,46504.93
Florida,2015,9,951.36,20491195.0,46427.75
Florida,2015,10,960.2,20525205.5,46781.5


In [176]:
%%sql SELECT DISTINCT year
FROM GDP_Percapita
ORDER BY year;

 * sqlite:////Users/emerino/Documents/Capstone_Project_Group_4-/Capstone.db
Done.


year
2015
2016
2017
2018
2019
2020
2021
2022
2023
2024


In [177]:
%%sql SELECT COUNT(DISTINCT state) AS num_states
FROM GDP_Percapita;

 * sqlite:////Users/emerino/Documents/Capstone_Project_Group_4-/Capstone.db
Done.


num_states
48


In [178]:
%%sql SELECT year,
       GROUP_CONCAT(DISTINCT month) AS months_present
FROM GDP_Percapita
GROUP BY year
ORDER BY year;

 * sqlite:////Users/emerino/Documents/Capstone_Project_Group_4-/Capstone.db
Done.


year,months_present
2015,"1,2,3,4,5,6,7,8,9,10,11,12"
2016,"1,2,3,4,5,6,7,8,9,10,11,12"
2017,"1,2,3,4,5,6,7,8,9,10,11,12"
2018,"1,2,3,4,5,6,7,8,9,10,11,12"
2019,"1,2,3,4,5,6,7,8,9,10,11,12"
2020,"1,2,3,4,5,6,7,8,9,10,11,12"
2021,"1,2,3,4,5,6,7,8,9,10,11,12"
2022,"1,2,3,4,5,6,7,8,9,10,11,12"
2023,"1,2,3,4,5,6,7,8,9,10,11,12"
2024,"1,2,3,4,5,6,7,8,9,10,11,12"
